# Lesson 09 Lab — Sampling and Output Control

**Puzzle:** Which parameter changed the answer when two requests used the same model?

This notebook retains the output of a complete RTX 5090 run.


## Why this matters

Sampling parameters are part of the public API, not harmless presentation options. Temperature changes the logit scale; top-p truncates the candidate mass; stop rules can remove suffixes; logprobs change response volume and observability.


## 0. Predict before running

1. Predict which cases are deterministic within one run.
2. Identify whether stop text appears in output.
3. State the extra payload cost of logprobs.

For every answer, name the observation that would disprove it.


## 1. Name the concrete objects

One native engine executes greedy, seeded stochastic, top-p, stop-string, and logprob cases. The result records token hashes and selected logprob metadata without treating variation as model quality.

- Temperature and top-p act at different stages of sampling.
- A fixed seed is necessary but not a cross-version guarantee.
- Stop rules affect both visible text and finish metadata.


## 2. Derive the mechanism

Greedy decoding selects the maximum logit. Temperature divides logits before normalization, while nucleus sampling keeps the smallest set whose cumulative probability reaches `top_p`. A seed scopes pseudo-random choices, but numerical and scheduling changes can still affect close probabilities. Stop conditions terminate generation after a matched token or string according to API semantics.

### Mechanism at a glance

```mermaid
flowchart LR
  L["model logits"] --> T["temperature scaling"]
  T --> P["top-p candidate set"]
  P --> R["seeded random draw"]
  R --> S{"stop matched?"}
  S -->|"no"| L
  S -->|"yes"| O["text + finish reason + optional logprobs"]
```

### Walk it step by step

1. **Freeze raw request settings.** Keep every sampling field beside the output.
2. **Separate selection stages.** Temperature rescales; top-p filters; the RNG draws.
3. **Inspect termination.** Read stop behavior and finish reason together.
4. **Evaluate quality elsewhere.** Variation is not evidence of improvement.


## 3. Inspect the execution environment

The next cell asserts CUDA, prints the RTX 5090/PyTorch/CUDA/vLLM identity, fixes a seed, and defines only the helpers used by this chapter.


In [1]:
LESSON_NO = 9
LESSON_TITLE = 'Sampling and Output Control'

from pathlib import Path
import gc, hashlib, importlib, inspect, ipaddress, json, math, os, random, re
import shutil, statistics, subprocess, sys, tempfile, time
from urllib.parse import urlparse

# The default FlashInfer sampler requires a local JIT link setup that is not
# guaranteed in wheel-only environments. vLLM's native PyTorch sampler keeps
# these labs reproducible without changing attention or scheduling backends.
os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")

import requests
import torch
import vllm
import yaml
from vllm import LLM, SamplingParams

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260812 + LESSON_NO
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
MODEL_PATH = os.environ.get("CH3_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")
MODEL = Path(MODEL_PATH)
assert MODEL.exists(), f"Set CH3_MODEL to a local model directory; not found: {MODEL}"

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name, "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__, "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0], "vllm": vllm.__version__,
    "model_path": MODEL.name, "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered: return float("nan")
    pos = (len(ordered) - 1) * q; lo, hi = math.floor(pos), math.ceil(pos)
    return ordered[lo] if lo == hi else ordered[lo] * (hi - pos) + ordered[hi] * (pos - lo)

def model_config():
    return json.loads((MODEL / "config.json").read_text(encoding="utf-8"))

def base_engine_args(**overrides):
    values = {
        "model": str(MODEL), "tokenizer": str(MODEL), "trust_remote_code": False,
        "dtype": "bfloat16", "max_model_len": 2048, "gpu_memory_utilization": 0.45,
        "enforce_eager": True, "seed": SEED, "max_num_seqs": 16,
    }
    values.update(overrides); return values

def output_record(item):
    completion = item.outputs[0]; tokens = list(completion.token_ids)
    return {
        "request_id": str(item.request_id), "prompt_tokens": len(item.prompt_token_ids or []),
        "output_tokens": len(tokens), "token_ids": tokens, "text_preview": completion.text[:120],
        "text_sha256": hashlib.sha256(completion.text.encode()).hexdigest(),
        "finish_reason": str(completion.finish_reason),
        "stop_reason": None if completion.stop_reason is None else str(completion.stop_reason),
        "num_cached_tokens": int(getattr(item, "num_cached_tokens", 0) or 0),
    }

def vllm_cli():
    candidate = Path(sys.executable).parent / "vllm"
    return str(candidate if candidate.exists() else (shutil.which("vllm") or "vllm"))

def cli_help(*args):
    result = subprocess.run([vllm_cli(), *args, "--help"], capture_output=True, text=True, timeout=60)
    return result.returncode, result.stdout + result.stderr

def run_server_probe(port, request_payload=None, scrape_metrics=False):
    log_path = Path(tempfile.gettempdir()) / f"ch03-vllm-{LESSON_NO}-{port}.log"
    command = [vllm_cli(), "serve", str(MODEL), "--host", "127.0.0.1", "--port", str(port),
               "--dtype", "bfloat16", "--max-model-len", "1024", "--gpu-memory-utilization", "0.45",
               "--enforce-eager", "--disable-uvicorn-access-log"]
    started = time.perf_counter()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, text=True)
    ready = False
    try:
        deadline = time.time() + 300
        while time.time() < deadline:
            if process.poll() is not None: break
            try:
                if requests.get(f"http://127.0.0.1:{port}/health", timeout=2).status_code == 200:
                    ready = True; break
            except requests.RequestException: pass
            time.sleep(1)
        startup_s = time.perf_counter() - started
        if not ready:
            raise RuntimeError("vLLM server failed to start:\n" + log_path.read_text(errors="replace")[-6000:])
        models = requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=30)
        data = {"server_ready": True, "startup_s": startup_s,
                "models_status": models.status_code, "model_json": models.json()}
        if request_payload is not None:
            tick = time.perf_counter()
            chat = requests.post(f"http://127.0.0.1:{port}/v1/chat/completions",
                                 json=request_payload, timeout=180)
            data.update(chat_status=chat.status_code, chat_latency_s=time.perf_counter() - tick,
                        chat_json=chat.json())
        if scrape_metrics:
            response = requests.get(f"http://127.0.0.1:{port}/metrics", timeout=30)
            data.update(metrics_status=response.status_code, metrics_text=response.text)
        return data
    finally:
        if process.poll() is None:
            process.terminate()
            try: process.wait(timeout=30)
            except subprocess.TimeoutExpired: process.kill(); process.wait(timeout=10)
        tail = log_path.read_text(errors="replace")[-4000:] if log_path.exists() else ""
        private_home = "/" + "root" + "/"
        globals()["SERVER_LOG_TAIL"] = tail.replace(str(MODEL), "$CH3_MODEL").replace(private_home, "<remote-home>/")


<remote-home>/vllm-ch03/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "python": "3.12.3",
  "vllm": "0.27.1",
  "model_path": "Qwen2.5-1.5B-Instruct",
  "seed": 20260821
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | greedy decoding with no stop rule |
| Candidate | seeded sampling, top-p, stop, and logprob variants |
| Held constant | model, prompt, maximum tokens, engine, and GPU |
| Measurements | token hashes, token counts, finish reasons, stop inclusion, and logprob presence |
| Evidence | `native-backend` |

**Experiment:** Run five explicit SamplingParams cases and compare token sequences, finish reasons, and logprob availability.


## 5. Inspect the experiment code

The code constructs a fresh SamplingParams object for every case and stores the effective settings next to its output. It never labels a different sample as better without an evaluator.

Do not execute until the code matches the frozen table.


In [2]:
llm=LLM(**base_engine_args(max_model_len=1024)); prompt="Write a sentence containing alpha, beta, and gamma."
params={"greedy":SamplingParams(temperature=0.0,max_tokens=24,seed=SEED),
        "sampled":SamplingParams(temperature=.8,top_p=1.0,max_tokens=24,seed=SEED),
        "top_p":SamplingParams(temperature=.8,top_p=.5,max_tokens=24,seed=SEED),
        "stop":SamplingParams(temperature=0.0,max_tokens=24,stop=["gamma"],seed=SEED),
        "logprobs":SamplingParams(temperature=0.0,max_tokens=12,logprobs=3,seed=SEED)}
rows={}
for name,setting in params.items():
    item=llm.generate([prompt],setting,use_tqdm=False)[0]; row=output_record(item)
    row["logprobs_returned"]=item.outputs[0].logprobs is not None; rows[name]=row
metrics={"cases":len(rows),"unique_hashes":len({x["text_sha256"] for x in rows.values()}),**rows}
metrics["logprobs"]={**metrics["logprobs"],"returned":metrics["logprobs"]["logprobs_returned"]}
analysis=(f"Five explicit configurations produced {metrics['unique_hashes']} hashes. Stop finished as "
          f"{metrics['stop']['finish_reason']} and logprobs returned={metrics['logprobs']['returned']}. "
          "Variation is localized to request parameters, not ranked as quality.")


INFO 08-13 00:18:40 [api_utils.py:273] non-default args: {'tokenizer': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct', 'dtype': 'bfloat16', 'seed': 20260821, 'max_model_len': 1024, 'gpu_memory_utilization': 0.45, 'max_num_seqs': 16, 'disable_log_stats': True, 'enforce_eager': True, 'model': '<remote-home>/autodl-tmp/models/Qwen/Qwen2.5-1.5B-Instruct'}


INFO 08-13 00:18:40 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 08-13 00:18:40 [model.py:1883] Using max model len 1024


INFO 08-13 00:18:40 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.


WARNING 08-13 00:18:40 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


...

(EngineCore pid=650619) INFO 08-13 00:18:52 [jit_monitor.py:79] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=650619) INFO 08-13 00:18:52 [core.py:355] init engine (profile, create kv cache, warmup model) took 2.78 s


(EngineCore pid=650619) WARNING 08-13 00:18:53 [vllm.py:1194] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=650619) WARNING 08-13 00:18:53 [vllm.py:1247] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=650619) INFO 08-13 00:18:53 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=650619) INFO 08-13 00:18:53 [vllm.py:1426] Cudagraph is disabled under eager mode
(EngineCore pid=650619) INFO 08-13 00:18:53 [compilation.py:329] Enabled custom fusions: norm_quant, act_quant


INFO 08-13 00:18:53 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


(EngineCore pid=650619) WARNING 08-13 00:18:54 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _topk_log_softmax_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; vLLM 0.27.1.

| Measured field | Checked-in value |
|---|---:|
| Cases | 5 |
| Unique token hashes | 5 |
| Greedy tokens | 24 |
| Sampled tokens | 24 |
| Stop finish reason | stop |
| Logprobs returned | yes |


## 7. Explain the result

Five explicit configurations produced 5 hashes. Stop finished as stop and logprobs returned=True. Variation is localized to request parameters, not ranked as quality.

This interpretation is bounded to the printed model, GPU, packages, workload, and evidence label.


## 8. Keep the evidence label honest

This run is labeled **`native-backend`**. The named vLLM runtime executed on the recorded GPU/model/workload. The result does not transfer to another version, model, endpoint, or traffic distribution.

The next cell writes and prints the canonical JSON artifact.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 9, "title": 'Sampling and Output Control', "environment": ENV,
    "evidence_label": 'native-backend', "metrics": metrics,
    "analysis": analysis, "conclusion": 'Sampling parameters define observable behavior; the experiment localizes output changes to explicit request configurations.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 9,
  "title": "Sampling and Output Control",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "python": "3.12.3",
    "vllm": "0.27.1",
    "model_path": "Qwen2.5-1.5B-Instruct",
    "seed": 20260821
  },
  "evidence_label": "native-backend",
  "metrics": {
    "cases": 5,
    "unique_hashes": 5,
    "greedy": {
      "request_id": "0",
      "prompt_tokens": 11,
      "output_tokens": 24,
      "token_ids": [
        24708,
        11,
        13440,
        11,
        323,
        21619,
        525,
        279,
        2326,
        6028,
        7987,
        315,
        3100,
        11,
        892,
        646,
        387,
        9519,
        311,
        1855,
        264,
        6884,
        2088,
        315
      ],
      "text_preview": " Alpha, beta, and gamma are the three primary colors of light, which can be mixed to create a wide range of",
     

## 9. Make the bounded decision

> Sampling parameters define observable behavior; the experiment localizes output changes to explicit request configurations.

**Acceptance/rollback:** Promote an API configuration only after deterministic, stochastic, stop, and observability cases match the product contract.

**Failure analysis:** Tokenizer boundaries can make a string stop behave differently from a token stop. Logprob structures and reproducibility guarantees can change across versions.


## 10. Extend the evidence

Add streaming stops, bad-word filters, min-p, repetition controls, and statistical distribution checks over many seeds.

The full boundary and references are in [`README.md`](README.md).
